In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import sys
sys.path.append("/content/drive/MyDrive/stocks/duet/duet_class")

from pipeline.config import DUETConfig
from pipeline import train, evaluate, predict
from pipeline.prepare_and_check import FinancialTimeSeriesPreparer
from pipeline.timefeatures import time_features_from_index
from pipeline.wf_slicer import GlobalNormConfig, SplitConfig, WalkForwardWindowSlicerVec, WindowConfig
from duet.model import DUETModel
from torch.utils.data import DataLoader, TensorDataset
import torch
import joblib
import random

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight


In [3]:
# Настройки
DATA_PATH = "/content/drive/MyDrive/stocks/Data/anomaly/anomaly_df_binance_BTCUSDT_futures_cufOff_1.5.joblib"
LABELS_PATH = "/content/drive/MyDrive/stocks/Data/anomaly/Dataset_2_7class_clear - Dataset_2_7class_clear.csv.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"

config = DUETConfig(
    # =========================
    # Общие параметры
    # =========================
    timestamp_col = "timestamp",
    features = [
       'Open', 'High', 'Low', 'Close',
       'Volume',
    ],                           # Название колонок для обучения
    forecast = 'target',          # Название колонки с таргетом
    not_to_normalise = [],       # Название колонок, которые НЕ НАДО нормализовать
    scaler = 'NONE',         # Тип нормализации (STD, MINMAX, QUANT)
    predict_type = 'detect',     # Детекция "detect" или предикт "next" следующей свечи
    seq_len = 48,                # Длина входной последовательности
    num_classes = 7,             # кол-во классов
    patch_len = 24,               # Длина патча (TCM)
    stride = 12,                  # Шаг между патчами (TCM)
    moving_avg = 25,              # Размер окна скользящего сглаживания

    K_t = 2,                    # Кол-во временных кластеров (TCM)
    K_c = 4,                    # Кол-во кластеров каналов (CCM)
    d_c = 32,                   # Размерность embedding каналов (CCM)
    top_k = 2,                  # Кол-во связей при разреживании маски
    use_revin = True,           # Включить RevIN/InstanceNorm (если отключить, то нужно нормализовать данные отдельно)
    revin_affine = True,        # Использовать affine параметры в RevIN
    revin_eps = 1e-5,           # Эпсилон для стабильности RevIN

    # =========================
    # Параметры модели
    # =========================
    d_model = 64,               # Размерность скрытого пространства в attention
    d_ff = 256,                 # Размерность feedforward слоя
    n_heads = 4,                # Количество голов в multi-head attention
    e_layers = 2,               # Количество слоев в encoder (CCM)
    dropout = 0.2,            # Dropout во всех слоях attention
    fc_dropout = 0.2,         # Dropout в выходном head слое
    activation = "relu",        # Активационная функция (relu, gelu, elu)
    num_experts = 4,            # Число экспертов (в Router, если используется)
    report_freq = 10,            # Частота появления confusion matrix

    # =========================
    # Режимы обработки
    # =========================
    CI = True,                 # Channel-Independent режим (если False — shared weights)
    use_router = True,          # Включить распределительный роутер
    timeenc = 1,                # Использовать time encoding (0 = без, 1 = sin/cos и т.п.)

    # =========================
    # Настройки обучения
    # =========================
    batch_size=32,          # можно немного увеличить при хорошем GPU
    epochs=300,             # больше эпох для более тонкой настройки
    learning_rate=1e-4,     # стандартный LR для Adam
    weight_decay=1e-5,
    patience=100,            # early stopping не слишком строгий

    # =========================
    # Прочее
    # =========================
    checkpoint_best = '/content/drive/MyDrive/stocks/duet/duet_class/weights/best_val_acc_weights_002.pt',       # Путь для сохранения лучших по val_accuracy весов
    checkpoint_final = '/content/drive/MyDrive/stocks/duet/duet_class/weights/final_weights_002.pt',      # Путь для сохранения финальных весов
    seed = 33,                  # Фиксированное зерно генератора случайных чисел
    verbose = True,            # Печать хода обучения
)

"""
Устанавливает seed для numpy, random, torch (вкл. CUDA).
Гарантирует воспроизводимость.
"""

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)

In [4]:
# --- 1. Предобработка данных ---
# Загрузка данных
df = joblib.load(DATA_PATH)
labels = pd.read_csv(LABELS_PATH)

data_preparer = FinancialTimeSeriesPreparer(
    tz="UTC",
    timestamp_col="timestamp",
    drop_warmup=True,
)
labels_preparer = FinancialTimeSeriesPreparer(
    tz="UTC",
    timestamp_col="Time_close",
    drop_warmup=True,
)

df, _ = data_preparer.prepare(df, ensure_ohlcv=True, from_date="2023-01-01", to_date="2024-01-30")
labels, _ = labels_preparer.prepare(labels, ensure_ohlcv=False, from_date="2022-01-01", to_date="2025-01-30")

labels.index = labels.index.round("1min")   # до ближайшей минуты

# Добавляем target, чтобы объединить метки аномалий с ценовым рядом
# и далее строить окна/предикты только по тем точкам, где есть валидная метка.
df['target'] = labels['target']
# Заполняем -1 для строк без метки: это нормальные/неразмеченные точки,
# для которых мы не строим предикт класса аномалии.
df.fillna({'target':-1}, inplace=True)
df['target'].value_counts()

time_features = time_features_from_index(df.index, timeenc=config.timeenc)
time_feature_names = [f"timeenc_{i}" for i in range(time_features.shape[1])]
df[time_feature_names] = time_features
config.features = config.features + time_feature_names

split = SplitConfig(
    n_folds=1,
    ratios=(1.0, 0.0, 0.0),
)

if config.predict_type.upper() == "DETECT":
    y_end_offset = 0
elif config.predict_type.upper() == "NEXT":
    y_end_offset = 1
else:
    raise ValueError("config.predict_type должен быть 'DETECT' или 'NEXT'")

window = WindowConfig(
    x_window=config.seq_len,
    x_end_offset=0,
    y_window=1,
    y_end_offset=y_end_offset,
    allow_left_context_for_x=False,
)

scaler_map = {"STD": "standard", "MINMAX": "minmax", "QUANT": "quantile", "NONE": "none"}
global_norm = GlobalNormConfig(scaler=scaler_map.get(config.scaler.upper(), "none"))

slicer = WalkForwardWindowSlicerVec(
    split=split,
    window=window,
    global_norm=global_norm,
    no_norm_cols=config.not_to_normalise,
    eps=1e-12,
    drop_incomplete_last_fold=True,
)

out = slicer.split_and_window(
    X=df[config.features],
    y=df[config.forecast],
    aux=df[['gaps', 'nan_filled', 'candle_error']]
)
fold0 = out["fold_0"]

x_windows = fold0["train"]["X"]
y_windows = fold0["train"]["y"][:, 0, 0].astype(int)
t0 = fold0["train"]["t0"]

# Оставляем только окна с реальной меткой класса (аномалией).
# target == -1 означает «нет аномалии/нет метки», поэтому предикт по ним не нужен.
valid_mask = y_windows != -1
x_windows = x_windows[valid_mask]
y_windows = y_windows[valid_mask]
t0 = t0[valid_mask]

# --- 6. Инициализация модели ---
model = DUETModel(config).to(device)

# Загрузка весов

# model.load_state_dict(torch.load(config.checkpoint_best)) # загрузка лучших весов
model.load_state_dict(torch.load(config.checkpoint_final)) # загрузка финальных весов
model.eval()


DUETModel(
  (revin): RevIN()
  (tcm): TCM(
    (decomp): SeriesDecomposition(
      (avg_pool): AvgPool1d(kernel_size=(25,), stride=(1,), padding=(12,))
    )
    (extractors): ModuleList(
      (0-1): 2 x LinearPatternExtractor(
        (projection): Linear(in_features=48, out_features=64, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
    )
    (router): Sequential(
      (0): Linear(in_features=24, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=2, bias=True)
    )
  )
  (ccm): CCM(
    (embedding): Linear(in_features=25, out_features=32, bias=True)
    (metric): Linear(in_features=32, out_features=32, bias=False)
  )
  (fusion): Fusion()
  (head): DUETHead(
    (head): Sequential(
      (0): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (1): Linear(in_features=64, out_features=64, bias=True)
      (2): ReLU()
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=64, out_features=7, bias

In [5]:
# Предикт одного окна
from pipeline.predict import predict_window

x_window = x_windows[0]
pred = predict_window(model, x_window, config, device=device)

print(pred)


[4.1086075e-04 1.2297048e-02 7.1243551e-03 4.5073393e-01 3.6723161e-01
 8.9751579e-02 7.2450653e-02]


In [6]:
from pipeline.predict import predict_dataset_batched

# Предсказания
y_pred, y_probs = predict_dataset_batched(model, x_windows, config, device="cuda")

print(f"Prediction labels array shape {y_pred.shape}")
print(f"Prediction probs array shape {y_probs.shape}")

# --- Слияние предиктов с исходным датасетом по индексу ---
# t0 хранит индексные позиции исходного ряда для каждого окна.
# В итоговом DataFrame оставляем все строки, а предикты заполняем только
# в позициях t0 (для валидных окон), остальные значения = -1.
pred_index = df.index[t0]
pred_series = pd.Series(-1, index=df.index, name="pred")
pred_series.loc[pred_index] = y_pred

df_with_pred = df.copy()
df_with_pred["pred"] = pred_series

df_with_pred[["target", "pred"]].head()


Prediction labels array shape (2009,)
Prediction probs array shape (2009, 7)


,target,pred
datetime,,
2023-01-01 00:00:00+00:00,-1.0,-1
2023-01-01 00:01:00+00:00,-1.0,-1
2023-01-01 00:02:00+00:00,-1.0,-1
2023-01-01 00:03:00+00:00,-1.0,-1
2023-01-01 00:04:00+00:00,-1.0,-1


In [10]:
mask = (df_with_pred["target"] != -1) & (df_with_pred["pred"] != -1)

if mask.any():
    percent = (df_with_pred.loc[mask, "target"] == df_with_pred.loc[mask, "pred"]).mean() * 100
else:
    percent = float("nan")  # или 0.0, если так принято

print(f" Совпадений: {percent}%")

 Совпадений: 69.98506719761075%
